# Learned Predictions for Shortest Path — Padova Road Network
**Progetto finale — Advanced Topics in Algorithms**
Nicole Mietto, Davide Sut

Notebook orchestratore: importa la logica da `src/`, `modelli/`,
`valutazione/` (vedi README.md per la struttura completa). Questo
notebook è intenzionalmente sottile — ogni cella chiama funzioni definite
nei moduli, senza duplicare logica al loro interno.


## 0. Setup ambiente (Colab o locale)

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/TUO-USERNAME/padova-routing.git"  # <-- aggiorna
    REPO_DIR = "/content/padova-routing"
    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    sys.path.insert(0, REPO_DIR)
    os.system(f"pip install -q -r {REPO_DIR}/requirements.txt")
else:
    sys.path.insert(0, ".")  # eseguito dalla root del repo

print(f"Ambiente: {'Colab' if IN_COLAB else 'locale'}")


In [ ]:
# Monta Drive (solo su Colab) e configura i path dei dati
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Configurazione percorsi ──────────────────────────────────────────────
# Struttura della cartella di progetto su Drive (vedi README per i dettagli):
#   grafi/                  -> grafi stradali grezzi (input, scaricati da OSM)
#   modelli_salvati/        -> modelli ML allenati (output del training)
#   traffico/               -> esperimenti di traffico per fascia oraria (non
#                              ancora integrati nella pipeline principale)
#   grafici e immagini/     -> output dei grafici prodotti dalle celle di analisi
#   archivio/cose vecchie/  -> notebook superati, mantenuti per riferimento
PROJECT_DIR = "/content/drive/MyDrive/Unipd/Advanced Topics in Algorithms" if IN_COLAB else "dati"

GRAFI_DIR = os.path.join(PROJECT_DIR, "grafi") if IN_COLAB else PROJECT_DIR
MODELLI_DIR = os.path.join(PROJECT_DIR, "modelli_salvati") if IN_COLAB else os.path.join(PROJECT_DIR, "modelli_salvati")
TRAFFICO_DIR = os.path.join(PROJECT_DIR, "traffico")
GRAFICI_DIR = os.path.join(PROJECT_DIR, "grafici e immagini")

GRAPHML_PATH = os.path.join(GRAFI_DIR, "padova_drive.graphml")
GRAPHML_VENETO_PATH = os.path.join(GRAFI_DIR, "veneto_drive.graphml")
MODEL_PATH_ANELLI = os.path.join(MODELLI_DIR, "modello_bcf_anelli.json")
MODEL_PATH_6ANELLI = os.path.join(MODELLI_DIR, "modello_bcf_6anelli.json")
MODEL_PATH_VENETO = os.path.join(MODELLI_DIR, "modello_bcf_veneto.json")
MODEL_PATH_STANDARD = os.path.join(MODELLI_DIR, "learned_potentials_model.joblib")

# I file di input/output BCF e i grafici prodotti a runtime restano in locale
# su Colab (non su Drive): sono temporanei o rigenerabili in pochi secondi.
BCF_INPUT_PATH = "/content/padova_bcf_input.txt" if IN_COLAB else "padova_bcf_input.txt"
BCF_DIR = "/content/negative_weight_SSSP" if IN_COLAB else "negative_weight_SSSP"

print(f"GRAFI_DIR    = {GRAFI_DIR}")
print(f"MODELLI_DIR  = {MODELLI_DIR}")
print(f"GRAPHML_PATH = {GRAPHML_PATH}")

## 1. Compilazione motore BCF

In [ ]:
from src.bcf import compila_bcf

BCF_BIN = compila_bcf(BCF_DIR)


## 2. Caricamento grafo

In [ ]:
from src.grafo import carica_ambiente

G, _ = carica_ambiente(GRAPHML_PATH)  # model=None: lo carichiamo a parte sotto


### 2.1 Traffico sintetico (opzionale)

Per default, i tempi di percorrenza (`travel_time`) vengono calcolati da OSMnx usando solo velocità massima e lunghezza dell'arco — un modello "liscio" senza variazioni realistiche di traffico.

`genera_traffico_realistico` applica una penalizzazione radiale (più traffico vicino al centro città) più una piccola componente di rumore casuale, per generare un dataset di training più realistico. **Da eseguire prima della generazione del dataset di training** (sezione 3), altrimenti i modelli si allenano sui tempi "lisci" originali.

Questa cella è disattivata di default (il traffico sintetico è un'aggiunta opzionale, non necessaria per il funzionamento base della pipeline) — decommenta per attivarla.

In [ ]:
from src.grafo import genera_traffico_realistico

APPLICA_TRAFFICO_SINTETICO = False  # cambia a True per attivare

if APPLICA_TRAFFICO_SINTETICO:
    G = genera_traffico_realistico(
        G,
        centro_lat=45.4064, centro_lon=11.8768,  # centro di Padova
        fattore_centro=1.5,      # +50% di tempo vicino al centro
        fattore_periferia=0.8,   # -20% di tempo in periferia
        raggio_centro_km=3.0,
        seed=42,
    )
    print("✅ Traffico sintetico applicato: G aggiornato in-place con i nuovi pesi.")
else:
    print("Traffico sintetico NON applicato (tempi OSMnx originali).")


## 3. Training dei modelli

Scegli quale modello allenare (o caricarne uno già salvato). Tutte le
funzioni sono in `modelli/`.

### 3.1 Modello ad anelli (3 fasce) — quello con i risultati migliori finora

In [ ]:
from modelli.base import WrapperXGBoost
from modelli.training_anelli import allena_modello_anelli, genera_dataset_anelli

CENTRO_LAT, CENTRO_LON = 45.4064, 11.8768
FASCE_KM = [(0, 5), (5, 15), (15, 60)]
NOMI_FASCE = ["Centro (0-5km)", "Periferia (5-15km)", "Provincia (15-60km)"]

# Decommenta per riallenare da zero (richiede qualche minuto):
# df_train_anelli = genera_dataset_anelli(
#     G, CENTRO_LAT, CENTRO_LON, FASCE_KM,
#     target_per_fascia=20, sorgenti_per_fascia_per_target=150,
#     nomi_fasce=NOMI_FASCE,
# )
# booster_anelli, feature_cols_anelli, metriche_anelli = allena_modello_anelli(
#     G, df_train_anelli, lambda_consistenza=0.5, n_round=300,
# )
# booster_anelli.save_model(MODEL_PATH_ANELLI)

# Altrimenti, carica il modello già allenato:
import xgboost as xgb
booster_anelli = xgb.Booster()
booster_anelli.load_model(MODEL_PATH_ANELLI)
feature_cols_anelli = ["node_lat", "node_lon", "target_lat", "target_lon", "haversine_dist_m"]

model_anelli = WrapperXGBoost(booster_anelli, feature_cols_anelli)
model = model_anelli  # modello attivo di default per le sezioni successive


## 4. Coppie source/target di test

In [ ]:
import requests, time

def coord_da_osm_id(osm_id: int) -> tuple[float, float]:
    url = f"https://www.openstreetmap.org/api/0.6/node/{osm_id}.json"
    r = requests.get(url, headers={"User-Agent": "padova-routing-project"})
    data = r.json()
    el = data["elements"][0]
    return el["lat"], el["lon"]

NODI_FISSI = {
    "Stazione FS": 5752101509,
    "Prato della Valle": 3360102756,
    "Università (Bo - Ponti Romani)": 611915608,
    "Arcella": 410121884,
    "Piazza Duomo": 605047784,
    "Aeroporto": 567948984,
    "Ospedale Civile": 7324269676,
}

PERCORSI = [
    ("Stazione FS", "Prato della Valle"),
    ("Università (Bo - Ponti Romani)", "Arcella"),
    ("Piazza Duomo", "Aeroporto"),
    ("Stazione FS", "Ospedale Civile"),
]

import osmnx as ox

def nearest_node_raggiungibile(G, lat, lon):
    import numpy as np
    nodes_df = ox.graph_to_gdfs(G, edges=False)
    dlat = nodes_df["y"] - lat
    dlon = nodes_df["x"] - lon
    nodes_df["dist"] = np.sqrt(dlat**2 + dlon**2)
    nodes_df = nodes_df.sort_values("dist")
    for nodo in nodes_df.index:
        if G.out_degree(nodo) > 1:
            return nodo
    return nodes_df.index[0]

coppie = []
for n_src, n_dst in PERCORSI:
    lat_s, lon_s = coord_da_osm_id(NODI_FISSI[n_src]); time.sleep(0.5)
    lat_d, lon_d = coord_da_osm_id(NODI_FISSI[n_dst]); time.sleep(0.5)
    src = nearest_node_raggiungibile(G, lat_s, lon_s)
    dst = nearest_node_raggiungibile(G, lat_d, lon_d)
    coppie.append((f"{n_src} → {n_dst}", src, dst))
    print(f"  {n_src} → {n_dst}:  src={src}  dst={dst}")


## 5. Verifica consistenza dell'euristica

Sanity check del segno, e verifica di consistenza a tre livelli (campione
casuale, percorso ottimale, nodi visitati da A*).

In [ ]:
from valutazione.consistenza import (
    sanity_check_segno,
    verifica_consistenza_campione,
    verifica_consistenza_percorso,
    verifica_consistenza_nodi_visitati,
)
from src.predizioni import genera_predizioni
from src.grafo import costruisci_archi_ridotti, sanifica_grafo
from src.bcf import esegui_bcf, esporta_per_bcf

_nome_test, _src_test, _dst_test = coppie[0]

sanity_check_segno(G, model, _dst_test)

y_hat, y_hat_int = genera_predizioni(G, model, _dst_test)
verifica_consistenza_campione(G, y_hat_int)
verifica_consistenza_percorso(G, y_hat_int, _src_test, _dst_test)

archi, nodo_to_idx, art_idx, _ = costruisci_archi_ridotti(G, y_hat_int)
esporta_per_bcf(archi, art_idx, BCF_INPUT_PATH)
phi, _ = esegui_bcf(BCF_BIN, BCF_INPUT_PATH, art_idx, len(G.nodes()))
G_san = sanifica_grafo(G, y_hat_int, phi, nodo_to_idx)

verifica_consistenza_nodi_visitati(G, G_san, y_hat_int, _src_test, _dst_test)


## 6. Confronto nodi esplorati: Dijkstra vanilla vs A*

In [ ]:
from valutazione.benchmark_nodi_esplorati import confronta_nodi_esplorati

df_confronto = confronta_nodi_esplorati(G, model, coppie, BCF_BIN, BCF_INPUT_PATH)
display(df_confronto)


## 7. Test su larga scala stratificato per fascia

In [ ]:
from src.grafo import classifica_per_fasce
from valutazione.benchmark_nodi_esplorati import (
    genera_coppie_stratificate,
    confronta_nodi_esplorati_stratificato,
    aggrega_per_fascia,
)

nodi_per_fascia = classifica_per_fasce(G, CENTRO_LAT, CENTRO_LON, FASCE_KM)
coppie_stratificate = genera_coppie_stratificate(
    nodi_per_fascia, NOMI_FASCE, list(G.nodes()), n_coppie_per_fascia=100
)

df_test_ampio = confronta_nodi_esplorati_stratificato(
    G, model, coppie_stratificate, BCF_BIN, BCF_INPUT_PATH
)
aggrega_per_fascia(df_test_ampio, NOMI_FASCE)


## 8. Sub-Graph Routing (rettangolo / ellisse)

Proposta dalla mail al prof: limitare le predizioni a un sottografo tra
source e target.

In [ ]:
from src.subgraph import extract_subgraph_bbox, extract_subgraph_ellipse, assicura_connessione

_nome_test, _src_test, _dst_test = coppie[0]

G_sub_bbox = extract_subgraph_bbox(G, _src_test, _dst_test, padding_km=2.0)
G_sub_bbox = assicura_connessione(G_sub_bbox, G, _src_test, _dst_test)
print(f"Bbox: {len(G_sub_bbox.nodes())} nodi (vs {len(G.nodes())} totali)")

G_sub_ellipse = extract_subgraph_ellipse(G, _src_test, _dst_test, padding_km=2.0)
G_sub_ellipse = assicura_connessione(G_sub_ellipse, G, _src_test, _dst_test)
print(f"Ellisse: {len(G_sub_ellipse.nodes())} nodi (vs {len(G.nodes())} totali)")


## 9. Interpolazione spaziale (Delaunay)

Proposta dalla mail al prof: predizioni ML solo su un campione di nodi
ancora, interpolate per i restanti.

In [ ]:
from src.interpolazione import genera_predizioni_interpolate

y_hat_interp, y_hat_int_interp = genera_predizioni_interpolate(
    G, model, _dst_test, sample_ratio=0.1, seed=42
)
print(f"Predizioni interpolate generate per {len(y_hat_int_interp)} nodi.")

# NOTA: verificare il segno con sanity_check_segno prima di usare in produzione
# (vedi avviso nel docstring di src/interpolazione.py)


---
## Note metodologiche

Vedi `README.md` per il riepilogo completo delle scelte di design.